Loader functions

In [0]:
import time, requests
from datetime import date, datetime, timedelta
from pathlib import Path
from zoneinfo import ZoneInfo

BASE    = "https://www.elprisetjustnu.se/api/v1/prices"
ROOT    = Path("/Volumes/laddstolpar_df/landing/raw/elpris")
ZONES   = ["SE1", "SE2", "SE3", "SE4"]
START   = date(2025, 10, 1)      # first quarter-hour day (decision 2026-09-23)
PAUSE_S = 0.2                    # no documented rate limit – be polite
TZ      = ZoneInfo("Europe/Stockholm")

def last_available_day(now=None):
    """Tomorrow's prices are published at 13:00 at the earliest."""
    now = now or datetime.now(TZ)
    return now.date() + timedelta(days=1) if now.hour >= 13 else now.date()

def target_path(d: date, zone: str) -> Path:
    # Deterministic: one file per zone and local day
    return ROOT / zone / f"{d:%Y}" / f"{d:%Y-%m-%d}_{zone}.json"

def fetch(url, tries=4):
    for attempt in range(1, tries + 1):
        r = requests.get(url, timeout=30)
        if r.status_code in (200, 404):            # 404 = not published (yet)
            return r
        if r.status_code == 429 or r.status_code >= 500:
            time.sleep(2 ** attempt)               # 2, 4, 8, 16 s
            continue
        r.raise_for_status()                       # other 4xx: fail loudly, no retry
    r.raise_for_status()

def land_elpris(start=START, end=None, zones=ZONES, max_calls=None):
    end = end or last_available_day()
    stats = {"calls": 0, "written": 0, "skipped": 0, "not_published": 0}
    d = start
    while d <= end:
        for z in zones:
            p = target_path(d, z)
            if p.exists() and p.stat().st_size > 0:   # idempotency: already landed
                stats["skipped"] += 1
                continue
            if max_calls is not None and stats["calls"] >= max_calls:
                return stats
            r = fetch(f"{BASE}/{d:%Y}/{d:%m-%d}_{z}.json")
            stats["calls"] += 1
            if r.status_code == 404:
                stats["not_published"] += 1
            else:
                r.json()                              # must be valid JSON before we write
                p.parent.mkdir(parents=True, exist_ok=True)
                p.write_text(r.text, encoding="utf-8")   # raw response, unchanged
                stats["written"] += 1
            time.sleep(PAUSE_S)
        d += timedelta(days=1)
    return stats

Call loader - small test run

In [0]:
test = dict(start=date(2026, 9, 20), end=date(2026, 9, 22))

print("Run 1:", land_elpris(**test))   # expect calls=12, written=12
print("Run 2:", land_elpris(**test))   # expect calls=0,  skipped=12

for p in sorted(ROOT.rglob("*.json")):
    print(p, p.stat().st_size, "bytes")

Call loader - full run

In [0]:
t0 = time.time()
stats = land_elpris()          # START=2025-10-01 → last_available_day()
elapsed = time.time() - t0

print(stats)
print(f"Elapsed: {elapsed/60:.1f} min, {elapsed/max(stats['calls'],1):.2f} s per call")
# Expect (before 13:00): calls=1424, written=1424, skipped=12, not_published=0

Data check

In [0]:
import json
from collections import Counter

from datetime import timezone

def quarter_hours_in_local_day(d: date) -> int:
    start = datetime(d.year, d.month, d.day, tzinfo=TZ).astimezone(timezone.utc)
    nxt = d + timedelta(days=1)
    end = datetime(nxt.year, nxt.month, nxt.day, tzinfo=TZ).astimezone(timezone.utc)
    return int((end - start).total_seconds() // 900)
    
end = last_available_day()
expected_days = [START + timedelta(days=i) for i in range((end - START).days + 1)]

problems, row_counts = [], Counter()
for z in ZONES:
    present = {p.name for p in (ROOT / z).rglob("*.json")}
    wanted  = {target_path(d, z).name for d in expected_days}
    missing, extra = sorted(wanted - present), sorted(present - wanted)
    if missing: problems.append(f"{z} missing {len(missing)}: {missing[:5]}")
    if extra:   problems.append(f"{z} unexpected {len(extra)}: {extra[:5]}")
    for d in expected_days:
        p = target_path(d, z)
        if p.exists():
            n = len(json.loads(p.read_text()))
            row_counts[n] += 1
            if n != quarter_hours_in_local_day(d):
                problems.append(f"{p.name}: {n} rows, expected {quarter_hours_in_local_day(d)}")

print(f"Days: {len(expected_days)}  ×  zones: {len(ZONES)}  =  {len(expected_days)*len(ZONES)} files expected")
print("Rows per file (rows: files):", dict(sorted(row_counts.items())))
print("Problems:", problems or "none")

In [0]:
%sql
CREATE VOLUME IF NOT EXISTS laddstolpar_df.ops.checkpoints
  COMMENT 'Auto Loader / streaming checkpoints';

Load into bronze

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.types import StructType, StructField, DoubleType, StringType

SRC    = "/Volumes/laddstolpar_df/landing/raw/elpris"
CHK    = "/Volumes/laddstolpar_df/ops/checkpoints/bronze_elpris"
TARGET = "laddstolpar_df.bronze.elpris"

# Explicit schema = the documented fields; anything unexpected lands in _rescued_data
schema = StructType([
    StructField("SEK_per_kWh", DoubleType()),
    StructField("EUR_per_kWh", DoubleType()),
    StructField("EXR",         DoubleType()),
    StructField("time_start",  StringType()),   # kept as raw string in bronze
    StructField("time_end",    StringType()),
])

q = (spark.readStream.format("cloudFiles")
        .option("cloudFiles.format", "json")
        .option("multiLine", "true")                 # each file is one JSON array
        .option("rescuedDataColumn", "_rescued_data")
        .schema(schema)
        .load(SRC)
        .select("*",
                F.col("_metadata.file_path").alias("_file"),
                F.col("_metadata.file_modification_time").alias("_file_modified"),
                F.current_timestamp().alias("_ingested_at"))
        .withColumn("elomrade", F.regexp_extract("_file", r"_(SE[1-4])\.json$", 1))
     .writeStream
        .option("checkpointLocation", CHK)
        .trigger(availableNow=True)                  # process what's there, then stop
        .toTable(TARGET))

q.awaitTermination()
print(q.lastProgress["numInputRows"] if q.lastProgress else "no progress info")

Check database

In [0]:
%sql
SELECT COUNT(*)                            AS rows_,
       COUNT(DISTINCT _file)               AS files,
       COUNT_IF(elomrade = '')             AS rows_without_zone,
       COUNT_IF(_rescued_data IS NOT NULL) AS rescued,
       COUNT_IF(SEK_per_kWh < 0)           AS negative_prices
FROM laddstolpar_df.bronze.elpris;

Table data for report

In [0]:
%sql DESCRIBE HISTORY laddstolpar_df.bronze.elpris